# 02 · Specialist Agents

What is an "agent", in code? Just four things:

| Part | Example |
|---|---|
| **name** | `"Orders agent"` |
| **instructions** | "You are the Orders specialist. Use lookup_order..." |
| **tools** | the list of tool descriptions the agent may use |
| **functions** | the real Python functions behind those tools |

We store them in a plain Python dictionary. One function, `run_agent`, can run **any** agent, because everything that makes the agents different is in the dictionary.

In [1]:
import json
from shop import client, MODEL     # the OpenAI client and the model name (see shop.py)

print("Ready")

Ready


## The tools
These are the tools from Modules 02 and 03, imported from `shop.py`.

In [2]:
from shop import LOOKUP_ORDER_TOOL, SEARCH_POLICIES_TOOL     # descriptions for the model
from shop import lookup_order, search_policies               # the real functions

print(lookup_order("A1002"))

{"item": "Running shoes", "price": 120.0, "status": "in_transit", "expected_delivery": "2026-11-25"}


## One function that runs any agent
This is the agent loop from Module 02: ask the model, run the tools it requests, repeat until it answers. The only change is that the instructions, tools and functions now come from the `agent` dictionary.

Read it from top to bottom. The `for` loop is the agent.

(`TOOL_RESULTS` keeps a log of what the tools returned. A reviewer agent will use it later in this module.)

In [3]:
TOOL_RESULTS = []  # a log of everything the tools returned, so a reviewer can check claims later


def call_tool(functions, name, args):
    """Run one tool and return its result as text."""
    if name not in functions:
        return "Unknown tool: " + name
    try:
        return str(functions[name](**args))
    except Exception as error:  # if a tool breaks, tell the model instead of crashing
        return "The tool failed: " + str(error)


def run_agent(agent, message, max_steps=6):
    """Give one message to an agent. It may use its tools several times, then it replies."""
    messages = [
        {"role": "system", "content": agent["instructions"]},
        {"role": "user", "content": message},
    ]

    # max_steps is a safety limit: every lap costs money, so never loop forever
    for step in range(max_steps):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=agent["tools"])
        reply = response.choices[0].message
        messages.append(reply)

        # No tool requested? The agent has its answer, so we are done.
        if not reply.tool_calls:
            return reply.content

        # Otherwise run each requested tool and give the result back to the agent
        for tool_call in reply.tool_calls:
            tool_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print("  [" + agent["name"] + "] " + tool_name + "(" + str(args) + ")")

            result = call_tool(agent["functions"], tool_name, args)
            TOOL_RESULTS.append(result)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return "Stopped: too many steps."

## Two specialists
Each has **one job** and **one tool**. Notice the instructions also say what they must **not** do. A good specialist knows its lane.

In [4]:
orders_agent = {
    "name": "Orders agent",
    "instructions": (
        "You are the Orders specialist at an online shop. "
        "You answer questions about a customer's order using the lookup_order tool. "
        "You do NOT answer questions about shop policies: say that a colleague handles those. "
        "Keep answers to 2 short sentences."
    ),
    "tools": [LOOKUP_ORDER_TOOL],
    "functions": {"lookup_order": lookup_order},
}

policy_agent = {
    "name": "Policy agent",
    "instructions": (
        "You are the Policy specialist at an online shop. "
        "Answer questions about shop rules ONLY from the results of the search_policies tool. "
        "If the answer is not there, say you don't know. "
        "You do NOT look up orders: say that a colleague handles those. "
        "Keep answers to 2 short sentences."
    ),
    "tools": [SEARCH_POLICIES_TOOL],
    "functions": {"search_policies": search_policies},
}

## Try each specialist on its own job

In [5]:
print("Customer: Where is my order A1002?")
print("Answer  :", run_agent(orders_agent, "Where is my order A1002?"))

print()
print("Customer: How long does express shipping take?")
print("Answer  :", run_agent(policy_agent, "How long does express shipping take?"))

Customer: Where is my order A1002?


  [Orders agent] lookup_order({'order_id': 'A1002'})


Answer  : Your order A1002 for running shoes is currently in transit. It is expected to be delivered on November 25, 2026.

Customer: How long does express shipping take?


  [Policy agent] search_policies({'question': 'How long does express shipping take?'})


Answer  : Express shipping takes 1 to 2 business days. Standard shipping takes longer, at 3 to 5 business days.


## What happens in the wrong lane?
Ask the Orders agent a policy question. Because of its instructions it should **stay in its lane** and not guess.

In [6]:
print("Customer: Do you ship internationally?   (asked to the Orders agent)")
print("Answer  :", run_agent(orders_agent, "Do you ship internationally?"))

Customer: Do you ship internationally?   (asked to the Orders agent)


Answer  : I'm unable to provide information about shop policies. A colleague can assist you with that.


That is useful, but it raises a question: **who decides which specialist gets the customer's message?** A person can't do it for every email. That is the next notebook.

## Recap
* An agent = **name + instructions + tools + functions**, stored in a dictionary.
* One `run_agent` function runs any of them.
* Specialists have a narrow job, few tools, and instructions that say what they must **not** do.